In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [7]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [8]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — can I still sign up now, or is it too late?',
 'answer_llm': 'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [9]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [10]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — can I still sign up now, or is it too late?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [11]:
assistant.reset_usage()

In [12]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [13]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [14]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [17]:
import pandas as pd
df_answers = pd.DataFrame(answers)
df_answers.head()

,question,answer_llm,answer_orig,document
0,I just found this course — can I still sign up...,"Yes, you can still join. If you want a certifi...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Am I allowed to join the course after it alrea...,"Yes, you can still join after the course has s...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I enroll late, do I still get access to eve...","Yes — if the submission form is still open, yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Can I still get a certificate if I join the co...,"Yes, but only if you submit your project while...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What do I need to do to make sure I’m eligible...,"To be eligible for the certificate, you need t...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [18]:
df_answers.to_csv("data/rag-answers-new.csv",index=False)

In [19]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [22]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [23]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [24]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [26]:
rec = answers[0]
rec

{'question': 'I just found this course — can I still sign up now, or is it too late?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [27]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [29]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

print(eval_result)

reasoning='The AI answer preserves the core meaning of the ground truth: enrollment is still possible, and certificate eligibility depends on submitting the project while submissions are open. Semantically equivalent.' score='good'


In [30]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [31]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point: you can still join, but to get a certificate you must submit the project while submissions are still open. This is semantically equivalent to the ground truth.', score='good')

In [32]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [33]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

In [34]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [36]:
df_eval = pd.DataFrame(evaluations)
df_eval

,question,document,score,reasoning
0,I just found this course — can I still sign up...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Am I allowed to join the course after it alrea...,74eb249bbf,good,The AI answer preserves the key meaning of the...
2,"If I enroll late, do I still get access to eve...",74eb249bbf,good,The AI answer captures the core idea that late...
3,Can I still get a certificate if I join the co...,74eb249bbf,good,The AI answer preserves the key point from the...
4,What do I need to do to make sure I’m eligible...,74eb249bbf,bad,The ground truth says late starters can still ...
...,...,...,...,...
510,Why does pip keep giving me requests 2.28 when...,4b30b918bc,good,The AI answer gives the same installation comm...
511,Is there a way to install requests straight fr...,4b30b918bc,good,The AI answer provides the exact install comma...
512,I’m hitting a 401 Client Error while setting t...,4b30b918bc,good,The AI answer matches the core idea of the gro...
513,What’s the exact pip command to install the ri...,4b30b918bc,good,The AI answer gives the exact pip command from...


In [37]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 495/515 = 96.12%


In [39]:
df_eval[df_eval["score"] == "bad"]

,question,document,score,reasoning
4,What do I need to do to make sure I’m eligible...,74eb249bbf,bad,The ground truth says late starters can still ...
7,Is there some approved student list that homew...,977bf7786c,bad,The ground truth clearly says there is no appr...
9,What should I do if I signed up but never got ...,977bf7786c,bad,The AI answer does not convey the ground truth...
29,"If I finish the materials later on my own, can...",69d122f12e,bad,The AI answer does not convey the ground truth...
34,What matters most for getting certified in thi...,9f689c185f,bad,The AI answer fails to answer the question and...
42,Is there a planned start for the course again?,bd31146b0e,bad,The ground truth says the course is planned to...
57,How do I find out when any live sessions for t...,d65e05bd7a,bad,The AI answer captures the key point that live...
123,What should I check if the API key still isn’t...,86d99bbf21,bad,The AI answer mentions one key check from the ...
136,What do I need in my .env file if I want to us...,0ae5c221b9,bad,The AI answer correctly identifies the key req...
157,What do I need to put in my shell startup file...,8b2f5e9d04,bad,"The AI answer captures the use of a .env file,..."


In [40]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)